In [2]:
import os

import pandas as pd
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt

from bluemath_tk.core.io import load_model

In [3]:
#### Load Data
pcs_df_with_kma = pd.read_csv(os.path.join("..", "..", "outputs", "alr", "pcs_with_kma.csv"))
pcs_df_with_kma = pcs_df_with_kma.rename(columns={'kma':'bmus', 'Unnamed: 0':'time'})

pcs_df_with_kma = pcs_df_with_kma.set_index('time')
pcs_df_with_kma.index = pd.to_datetime(pcs_df_with_kma.index)

In [4]:
kma_ob = load_model('/lustre/geocean/WORK/users/alonsoap/projects/TWINPEACS/02_Emulator_Testing/wt_var_testing/outputs/alr/classifier.pkl')

In [5]:
#### Load data from ALR_KDE
data_sim = xr.open_dataset('/lustre/geocean/WORK/users/alonsoap/projects/TWINPEACS/02_Emulator_Testing/wt_var_testing/outputs/model_comp/alr_kde.nc')

In [6]:
results = []
pc_cols = ['PC1','PC2','PC3','PC4','PC5','PC6']

for s in data_sim.seed.values:
    sim_df = data_sim.isel(seed=s)['pcs'].to_pandas()
    kma_predict, _ = kma_ob.predict(sim_df[pc_cols])

    sim_df['bmus_alr']     = data_sim.isel(seed=s).bmus.values
    sim_df['bmus_reclass'] = kma_predict.kma_bmus.values
    sim_df['seed']         = s
    results.append(sim_df[['seed', 'bmus_alr', 'bmus_reclass']])

check = pd.concat(results)
check['match'] = check.bmus_alr == check.bmus_reclass

print(f"overall: {check.match.mean():.1%}")

overall: 71.6%
